In [1]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

# ---------- USER INPUT ----------
r_slr_only = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_wo_VLM_stat.tif"   # depth due to SLR only (no VLM)
r_slr_vlm  = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"                     # depth due to SLR + VLM
out_path   = r"D:\Phd Research\Final_Raster\VLM_only_depth_100yr_compound.tif"
TOL = 0.0     # keep signed differences; set e.g. 0.05 to ignore tiny changes
LIMIT = 12.0  # optional cap for sanity checks
# ---------------------------------

# Read SLR+VLM as reference grid
with rasterio.open(r_slr_vlm) as ref:
    slr_vlm = ref.read(1).astype(float)
    ref_profile = ref.profile
    ref_crs = ref.crs
    ref_transform = ref.transform
    nodata_ref = ref.nodata if ref.nodata is not None else -9999.0

# Read SLR-only and reproject to reference grid if needed
with rasterio.open(r_slr_only) as src:
    slr_only_raw = src.read(1).astype(float)
    nodata_slr = src.nodata if src.nodata is not None else -9999.0
    same_grid = (src.crs == ref_crs) and (src.transform == ref_transform) and \
                (src.width == ref_profile['width']) and (src.height == ref_profile['height'])

    if same_grid:
        slr_only = slr_only_raw
    else:
        slr_only = np.full(slr_vlm.shape, nodata_slr, dtype=float)
        reproject(
            source=slr_only_raw,
            destination=slr_only,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            src_nodata=nodata_slr,
            dst_nodata=nodata_slr,
            resampling=Resampling.bilinear  # depths are continuous
        )

# Build mask of valid overlapping cells
mask = (
    np.isfinite(slr_vlm) & np.isfinite(slr_only) &
    (slr_vlm != nodata_ref) & (slr_only != nodata_slr)
)

# Compute VLM-only depth: (SLR+VLM) − (SLR only)
vlm_only = np.full_like(slr_vlm, nodata_ref, dtype=float)
vlm_only[mask] = slr_vlm[mask] - slr_only[mask]

# Optional: apply sanity filters (comment out if not needed)
valid = (vlm_only >= -LIMIT) & (vlm_only <= LIMIT)
vlm_only[~valid & mask] = nodata_ref

# Optional: ignore tiny changes (within ±TOL) by setting to 0 (or nodata)
if TOL > 0:
    near_zero = mask & (np.abs(vlm_only) < TOL)
    vlm_only[near_zero] = 0.0  # or set to nodata_ref if you prefer to drop them

# Save GeoTIFF
profile_out = ref_profile.copy()
profile_out.update(dtype="float32", nodata=nodata_ref)
with rasterio.open(out_path, "w", **profile_out) as dst:
    dst.write(vlm_only.astype("float32"), 1)

# Quick stats
vals = vlm_only[(vlm_only != nodata_ref) & np.isfinite(vlm_only)]
if vals.size:
    pct_pos = 100.0 * np.sum(vals > 0) / vals.size   # VLM increases depth
    pct_neg = 100.0 * np.sum(vals < 0) / vals.size   # VLM decreases depth
    print("=== VLM-only Depth (SLR+VLM − SLR-only) ===")
    print(f"Output saved to: {out_path}")
    print(f"Valid cells: {vals.size:,}")
    print(f"Mean = {np.mean(vals):.3f} m | Median = {np.median(vals):.3f} m")
    print(f"> 0 m (VLM deepens):  {pct_pos:.2f}%")
    print(f"< 0 m (VLM shallows): {pct_neg:.2f}%")
else:
    print("No valid overlapping cells after masking.")


=== VLM-only Depth (SLR+VLM − SLR-only) ===
Output saved to: D:\Phd Research\Final_Raster\VLM_only_depth_100yr_compound.tif
Valid cells: 353,116
Mean = 0.141 m | Median = 0.141 m
> 0 m (VLM deepens):  96.40%
< 0 m (VLM shallows): 3.51%
